# Dependencies

In [1]:
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes wandb
# !pip install datasets
# !pip install websockets nest_asyncio

In [2]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mukundnjoy (mukundnjoy-paypal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Physics Engine & Constants

In [3]:
"""
Heat Treatment GRPO Reward Function (V6) — WebSocket Fix.

ROOT CAUSE: OpenEnv's HTTP /reset and /step are STATELESS — each call creates
a fresh environment, steps once, destroys it. Temperature never accumulated.

FIX: Use WebSocket /ws endpoint (stateful session per connection).

REQUIREMENTS (add to Colab install cell):
    !pip install websockets nest_asyncio

Copy this into TRL.ipynb Cell 2 (the reward function cell).
"""

import re
import json
import asyncio
import nest_asyncio
import wandb
from websockets.asyncio.client import connect as ws_connect

nest_asyncio.apply()  # Allow nested event loops in Colab/Jupyter

HF_WS_URL = "wss://mukundnjoy-heat-treatment-scheduler.hf.space/ws"

TASK_NAME = "easy-bake"

ALLOY_CONSTANTS = {
    "easy-bake": {
        "temp_max": 600.0,
        "r_max_clip": 30.0,
        "r_target": 12.5,
        "r_target_min": 10.0,
        "r_target_max": 15.0,
        "growth_threshold": 176.0,
        "temp_melt": 502.0,
    },
    "medium-bake": {
        "temp_max": 1500.0,
        "r_max_clip": 16.0,
        "r_target": 6.5,
        "r_target_min": 5.0,
        "r_target_max": 8.0,
        "growth_threshold": 490.0,
        "temp_melt": 1400.0,
    },
    "hard-bake": {
        "temp_max": 1800.0,
        "r_max_clip": 50.0,
        "r_target": 22.5,
        "r_target_min": 20.0,
        "r_target_max": 25.0,
        "growth_threshold": 560.0,
        "temp_melt": 1600.0,
    },
}

CONST = ALLOY_CONSTANTS[TASK_NAME]


async def _run_episode_ws(text: str) -> dict:
    """Run one full episode over a WebSocket connection (stateful)."""

    matches = re.findall(r"^([0-5])\s*,\s*(\d+(?:\.\d+)?)$", text, re.MULTILINE)

    if not matches:
        return {"reward": -500.0, "error": "no_valid_actions"}

    try:
        async with ws_connect(HF_WS_URL, open_timeout=15, close_timeout=5) as ws:
            # === RESET ===
            await ws.send(json.dumps({"type": "reset", "data": {"task_name": TASK_NAME}}))
            reset_resp = json.loads(await asyncio.wait_for(ws.recv(), timeout=10))

            if reset_resp.get("type") == "error":
                return {"reward": -500.0, "error": f"reset_error: {reset_resp}"}

            total_reward = 0.0
            steps_taken = 0
            final_temp = 0.0
            final_r = 0.0
            max_temp_reached = 0.0
            prev_temp = 20.0
            prev_r = 0.0
            entered_growth_phase = False

            for match in matches:
                action_num = int(match[0])
                safe_duration = min(max(float(match[1]), 1.0), 600.0)

                if action_num == 5:  # Terminate
                    break

                # === STEP === (data contains raw action fields)
                await ws.send(json.dumps({
                    "type": "step",
                    "data": {
                        "action_num": action_num,
                        "duration_minutes": safe_duration
                    }
                }))
                step_resp = json.loads(await asyncio.wait_for(ws.recv(), timeout=15))

                if step_resp.get("type") == "error":
                    total_reward -= 100.0
                    break

                data = step_resp.get("data", {})
                total_reward += data.get("reward", 0.0) or 0.0
                steps_taken += 1

                obs = data.get("observation", {})
                final_temp = obs.get("temperature", 0.0) * CONST["temp_max"]
                final_r = obs.get("radius", 0.0) * CONST["r_max_clip"]

                if final_temp > max_temp_reached:
                    max_temp_reached = final_temp

                # --- Per-step breadcrumbs ---
                temp_delta = max(final_temp - prev_temp, 0.0)
                if final_temp < CONST["growth_threshold"]:
                    total_reward += temp_delta * 1.0  # +1/°C toward growth
                elif final_temp < CONST["temp_melt"] - 100:
                    total_reward += 5.0  # In growth zone hold bonus

                r_delta = max(final_r - prev_r, 0.0)
                if r_delta > 0:
                    total_reward += r_delta * 20.0  # +20 per nm grown

                prev_temp = final_temp
                prev_r = final_r

                if final_temp >= CONST["growth_threshold"]:
                    entered_growth_phase = True

                if data.get("done", False):
                    break

            # === END-OF-EPISODE SHAPING ===

            # Phase-gate (dominant signal)
            if entered_growth_phase:
                total_reward += 150.0
            else:
                total_reward -= 150.0

            # Min-steps penalty (anti-quit hack)
            if steps_taken < 5:
                total_reward -= (5 - steps_taken) * 30.0

            # Radius progress (only if didn't melt)
            if final_temp < CONST["temp_melt"]:
                progress = min(final_r / CONST["r_target"], 1.0)
                total_reward += progress * 200.0
                if final_r >= CONST["r_target_min"]:
                    total_reward += 50.0
                if CONST["r_target_min"] <= final_r <= CONST["r_target_max"]:
                    total_reward += 100.0

            # Clamp total reward to prevent float overflow in GRPO
            total_reward = max(-500.0, min(500.0, total_reward))

            return {
                "reward": total_reward,
                "radius_nm": final_r,
                "core_temp_C": final_temp,
                "max_temp_C": max_temp_reached,
                "recipe_steps": steps_taken,
                "entered_growth": entered_growth_phase,
            }

    except Exception as e:
        return {"reward": -500.0, "error": str(e)}


def api_physics_reward_func(completions, **kwargs):
    """GRPO reward: runs each completion as a stateful WebSocket episode."""
    rewards = []

    loop = asyncio.get_event_loop()

    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        result = loop.run_until_complete(_run_episode_ws(text))

        rewards.append(result["reward"])

        wandb.log({
            "physics/radius_nm": result.get("radius_nm", 0.0),
            "physics/core_temp_C": result.get("core_temp_C", 0.0),
            "physics/max_temp_C": result.get("max_temp_C", 0.0),
            "physics/recipe_steps": result.get("recipe_steps", 0),
            "physics/reward": result["reward"],
            "physics/entered_growth": 1.0 if result.get("entered_growth") else 0.0,
        })

        if result.get("error"):
            print(f"  ⚠ Episode error: {result['error']}")

    return rewards


# Inference & Evaluation Loop

In [4]:
import re
import wandb
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import GRPOConfig, GRPOTrainer


# system_prompt = """You are an expert Metallurgical Process Engineer controlling a precipitation hardening furnace.

# Your objective: Grow nanoprecipitates to the exact target radius (22.5 nm) without melting the material.

# CRITICAL PHYSICS CONSTRAINTS:
# 1. The material starts at 20°C. Precipitates WILL NOT GROW until the core temperature reaches ~600°C.
# 2. Your maximum heating action (4) only increases the furnace by +50°C per step.
# 3. You have a strict time limit! Do not waste the clock during the heating phase.

# STRATEGY:
# Use very short durations (e.g., 2 minutes) to rapidly stack +50°C furnace increases. Once the furnace is extremely hot, use a long hold duration (e.g., 120 minutes) to allow the massive casting's core temperature to catch up and thaw.

# Format each step on a new line as exactly TWO numbers separated by a comma: [Action_Num, Duration_Minutes]

# ACTION_NUM (0-4):
#     0: Aggressive cooling (-50°C)
#     1: Gentle cooling (-10°C)
#     2: Hold furnace temperature (0°C change)
#     3: Gentle heating (+10°C)
#     4: Aggressive heating (+50°C)

# DURATION_MINUTES: (1.0 to 600.0)

# EXAMPLE RECIPE:
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 2, 120.0
# 0, 30.0

# Respond with ONLY your sequence of steps."""

# system_prompt = """You are an expert Metallurgical Process Engineer controlling a precipitation hardening furnace.

# Your objective: Grow nanoprecipitates to the exact target radius (12.5 nm) without melting the material.

# CRITICAL PHYSICS CONSTRAINTS:
# 1. The material starts at 20°C. Precipitates WILL NOT GROW until the core temperature reaches ~176°C.
# 2. Your maximum heating action (4) increases the furnace by +50°C per step.
# 3. WARNING: The material melts at 502°C! Stay well below this temperature.

# STRATEGY:
# Heat aggressively with action 4 and short durations (2 min) until the furnace is around 300-350°C (safely in the growth phase). Then hold with action 2 and long durations (60-120 min) to let precipitates grow. Monitor the radius — when it approaches 12.5 nm, cool down with action 1 or 0.

# Format each step on a new line as exactly TWO numbers separated by a comma: [Action_Num, Duration_Minutes]

# ACTION_NUM (0-5):
#     0: Aggressive cooling (-50°C)
#     1: Gentle cooling (-10°C)
#     2: Hold furnace temperature (0°C change)
#     3: Gentle heating (+10°C)
#     4: Aggressive heating (+50°C)
#     5: TERMINATE EPISODE (use when radius is at target)

# DURATION_MINUTES: (1.0 to 600.0)

# EXAMPLE RECIPE:
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 4, 2.0
# 2, 120.0
# 1, 30.0
# 5, 1.0

# Respond with ONLY your sequence of steps."""

# dummy_data = {"prompt": [[
#     {"role": "system", "content": system_prompt},
#     {"role": "user", "content": "Context: Al-2024 in Lab Scale. Target: 12.500 nm. Generate your complete thermal recipe."}
# ]] * 200}
# dataset = Dataset.from_dict(dummy_data)

system_prompt = """You are an expert Metallurgical Process Engineer controlling a precipitation hardening furnace.
Your objective: Grow nanoprecipitates to the exact target radius (12.5 nm) without melting the material.
CRITICAL PHYSICS CONSTRAINTS:
1. The material starts at 20°C. Precipitates WILL NOT GROW until the core temperature reaches ~176°C.
2. Your maximum heating action (4) increases the furnace by +50°C per step.
3. WARNING: The material melts at 502°C! Stay well below this temperature.
STRATEGY:
Heat aggressively with action 4 and short durations (2 min) until the furnace is around 300-350°C (safely in the growth phase). Then hold with action 2 and long durations (60-120 min) to let precipitates grow. Monitor the radius — when it approaches 12.5 nm, cool down with action 1 or 0.
Format each step on a new line as exactly TWO numbers separated by a comma: [Action_Num, Duration_Minutes]
ACTION_NUM (0-5):
    0: Aggressive cooling (-50°C)
    1: Gentle cooling (-10°C)
    2: Hold furnace temperature (0°C change)
    3: Gentle heating (+10°C)
    4: Aggressive heating (+50°C)
    5: TERMINATE EPISODE (use when radius is at target)
DURATION_MINUTES: (1.0 to 600.0)
EXAMPLE RECIPE:
4, 2.0
4, 2.0
4, 2.0
4, 2.0
4, 2.0
4, 2.0
2, 120.0
1, 30.0
5, 1.0
Respond with ONLY your sequence of steps."""
dummy_data = {"prompt": [[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Context: Al-2024 in Lab Scale. Target: 12.500 nm. Generate your complete thermal recipe."}
]] * 200}
dataset = Dataset.from_dict(dummy_data)


# Model --> Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(model, r=16, target_modules=["q_proj", "v_proj"])

# Training --> TRL (Huggingface)
training_args = GRPOConfig(
    output_dir="grpo_digital_twin",
    learning_rate=5e-6,
    report_to="wandb",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_prompt_length=256,
    max_completion_length=512,
    num_generations=4, # Explores 4 different paths per iteration
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=[api_physics_reward_func],
    train_dataset=dataset,
    tokenizer=tokenizer,
)

# 6. Initialize WandB Run and Start Training
wandb.init(project="heat-treatment-grpo", name="predictive-braking-live")
print("🔥 Initiating Continuous ODE Physics Verification...")
trainer.train()
wandb.finish()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.8 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


🔥 Initiating Continuous ODE Physics Verification...


...

  ⚠ Episode error: no_valid_actions


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  ⚠ Episode error: no_valid_actions


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

physics/core_temp_C,▃▁▁▄▁█▆▁▆▆▅██▁▄▅██▁▁▇█▅█▆▄█▆█▇█▃▅▂▅█▆▃█▆
physics/entered_growth,▁████▁█▁█████▁█████████▁████▁███████████
physics/max_temp_C,▅▅█▅▂▅▂▆▅▁▅▁█▁█████▆██▅▆▅▅▄█▅██▆▄▅▁▅▆▂█▅
physics/radius_nm,█▁█▁▁█████▇▁█████████▁▄█████████▅███████
physics/recipe_steps,▁▁▁▃▁▅▄▃▄▂▃▅▄▄▁▅▁▅▁▃▅▄▅█▄▅▅▃▃▃▃▃▃▄▄▃▃▃▃▄
physics/reward,█▁▄██▆█▃▄▃███████▁██▇▁███████▄██████████
profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▂▁▂▁▂▂▄▄▂▂▄▄▃▄▃▅▄▄▂▃▃▃▄▄█▅▄▅▆▄▃▃▄▂▄▄▄▃▅▄
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▁▆▁▁▁▁▁▁▄▁▁▆▁▁▁▁▁▁▁▁▁▁▁▁▆▁▁▁▁▁▁▆▁▁█▁█▁▁▁
profiling/Time taken: UnslothGRPOTrainer.api_physics_reward_func,▁▃▂▁▃▃▂▄▂▄▄▃▄▃▂▃▂▃▄▄▃▂▄▃▃▃▄█▄▄▄▄▄▃▃▃▃▅▂▃
profiling/Time taken: UnslothGRPOTrainer.transformers.generate,▄▃▂▁██▆▁▂▂▂▃▃█▇█▄▆█▃▂▃▃█▆▁▃▂▂█▁▅▇█▄█▂▆▁▇
+25,...


In [4]:
model.save_pretrained("grpo_digital_twin/checkpoint-330")
tokenizer.save_pretrained("grpo_digital_twin/checkpoint-330")
print("✅ Checkpoint saved")